# Assignment 8 — Solution

## Embedding and retrieval design for an international law firm

**Module 2 · Lesson 8 — Embedding Models Masterclass**  
**Author:** Naeem Naseer

---

### Requirements

| Requirement | Value |
| --- | --- |
| Corpus | 15 million legal documents |
| Languages | English, French, German, Arabic |
| End-to-end latency | Under 2 seconds |
| Retrieval quality | High |
| Deployment | Cloud |

The design below uses **hybrid first-stage retrieval followed by cross-encoder reranking**. This preserves exact legal identifiers and citations while also finding semantically equivalent language.

## 1. Proposed architecture

### Offline ingestion

```text
15M legal documents
        │
        ▼
Object storage + metadata/ACL registry
        │
        ▼
Parse/OCR → language detection → Unicode normalization
        │
        ▼
Structure-aware legal chunking (court, section, clause, citation)
        │
        ├──► multilingual bi-encoder ──► dense vector index
        │
        └──► legal text + identifiers ──► BM25/sparse index
```

### Online query path

```text
User query
    │
    ▼
Authentication + language detection + query normalization
    │
    ├──► dense ANN retrieval ───────┐
    │                               ├──► fusion + ACL filter ──► top ~97
    └──► BM25/sparse retrieval ─────┘                              │
                                                                  ▼
                                                multilingual cross-encoder
                                                                  │ top 8–12
                                                                  ▼
                                                context builder → LLM → answer
                                                                  │
                                                                  ▼
                                                cited sources + audit trace
```

ACL filtering must happen before any text is exposed to reranking or generation. Each result retains document ID, jurisdiction, date, language, section, embedding version, and access-control metadata.

## 2. Core questions

### Q1 — Which embedding model family would I evaluate first, and why?

I would evaluate **BGE-M3 first**, then compare it with `multilingual-e5` and a suitable Jina multilingual model on the firm's own data.

BGE-M3 is a strong first candidate because this system needs:

- one shared multilingual space for English, French, German, and Arabic;
- cross-language retrieval, such as an Arabic question finding an English judgment;
- dense semantic matching plus support for sparse/exact-term signals;
- a model that can be hosted in the cloud and benchmarked without depending only on a public leaderboard.

This is only a **shortlist decision**, not the final selection. MTEB scores cannot represent the firm's legal vocabulary, Arabic documents, OCR noise, latency, licensing constraints, or actual query distribution. The winner must be chosen using a legal-domain golden set.

Before indexing, I would verify the selected model's required query/document prefixes and maximum input length. Missing prefixes or silently truncated chunks can destroy quality without producing an error.

### Q2 — Bi-encoder, cross-encoder, or both?

**Both, in two stages.**

1. A multilingual **bi-encoder** creates reusable document vectors offline. At query time, ANN search reduces roughly 120 million chunks to about 100 candidates in milliseconds.
2. A multilingual **cross-encoder** jointly reads each query–candidate pair and reranks only that small candidate set.

A cross-encoder cannot replace corpus retrieval because its score belongs to a query–document **pair**. Document-side scores therefore cannot be precomputed. Under the lesson's representative timings, scoring all 120 million chunks at 8 ms each would take about **267 hours for one query**, around **480,000 times the entire two-second budget**. A bi-encoder is fast enough at corpus scale but loses some fine-grained legal distinctions, so the cross-encoder supplies precision after retrieval.

The initial retriever should be hybrid: dense ANN catches semantic matches, while BM25 or learned sparse retrieval catches case numbers, statute identifiers, quoted phrases, names, and rare legal terms. Reciprocal-rank fusion can merge both lists before reranking.

In [ ]:
# Why a cross-encoder cannot score the whole corpus at query time
documents = 15_000_000
chunks_per_document = 8
chunks = documents * chunks_per_document
cross_encoder_ms_per_pair = 8
whole_corpus_hours = chunks * cross_encoder_ms_per_pair / 1000 / 60 / 60
times_over_budget = chunks * cross_encoder_ms_per_pair / 2_000

print(f"Chunks: {chunks:,}")
print(f"Whole-corpus cross-encoding: {whole_corpus_hours:,.1f} hours/query")
print(f"Multiple of 2-second budget: {times_over_budget:,.0f}x")

### Q3 — How would I benchmark embedding models?

I would use this repeatable evaluation process:

1. Build a versioned **golden set** from real, de-identified legal searches. It should contain queries, graded relevant chunks, and hard negatives that concern the same law or case but answer a different issue.
2. Stratify it by language, language pair, jurisdiction, document type, query type, document age, OCR quality, and easy versus difficult cases. Include both same-language and cross-language queries.
3. Keep the chunker, corpus snapshot, ANN settings, filters, and test hardware fixed so that only the embedding model changes.
4. Use exact search on a manageable subset to separate **embedding quality** from ANN index recall. Tune ANN parameters independently.
5. Measure Recall@10/50/100, MRR, nDCG@10, Precision@k, p50/p95/p99 latency, throughput, memory, indexing time, and cost per query.
6. Evaluate the complete pipeline too: dense-only, sparse-only, hybrid, and hybrid plus reranker. A model that wins alone may not win in the production pipeline.
7. Run paired significance tests or confidence intervals, inspect failures with legal subject-matter experts, then shadow-test and A/B-test the best candidates.

No test split should overlap the model-tuning set. Relevance judgments should be reviewed for disagreement, especially across languages.

### Q4 — Which metrics would I monitor in production?

| Area | Metrics |
| --- | --- |
| Retrieval quality | Recall@k, MRR, nDCG@k, Precision@k on a continuously replayed judged set |
| User outcomes | citation click-through, reformulation rate, no-result rate, helpfulness, escalation rate |
| Grounding | answer citation coverage, citation correctness sampled by reviewers, unsupported-answer rate |
| Latency | end-to-end and per-stage p50/p95/p99: embedding, ANN/BM25, reranking, LLM |
| Reliability | timeout/error rate, empty retrievals, index freshness lag, ingestion failures |
| Scale and cost | queries/second, tokens/query, GPU/CPU utilization, cache hit rate, cost/query |
| Segments and drift | every quality/latency metric split by language, jurisdiction, query type, tenant, and model/index version |
| Security | ACL-filter violations (target: zero), denied-result counts, audit completeness |

Aggregate averages can hide a severe Arabic regression, so dashboards and alerts must be segmented. Online behavior is useful but not a perfect relevance label; therefore it should be combined with a stable, human-judged evaluation set.

### Q5 — How would I support multiple languages?

- Detect and store language at document and chunk level, while allowing mixed-language legal text.
- Normalize Unicode carefully, preserve legally meaningful punctuation and citations, and use language-aware OCR and parsing.
- Embed all four languages with the **same multilingual model and version**, using the exact query/passsage prefix convention required by that model.
- Maintain both same-language and cross-language golden tests for every required direction—not only English queries.
- Use multilingual sparse analyzers and character-aware matching for identifiers and names alongside dense retrieval.
- Use a multilingual cross-encoder, or route to validated language-specific rerankers if the benchmark proves they are materially better.
- Consider query translation as a measured fallback or an additional retrieval branch, not as an assumed replacement: translation can alter legal terminology.
- Choose chunk sizes by model **tokens**, not characters. Arabic tokenization may fit less content into the same token limit.

If most traffic is English, I would benchmark a routed design—strong English model plus multilingual handling—against one shared multilingual model. The extra operational complexity is justified only if measured quality improves.

## 3. Harder questions

### Q6 — RAM for 15M documents at 1024 dimensions

**Assumption:** each document produces an average of **8 chunks**, matching the lesson's estimate of 120 million chunks. Each chunk has one 1024-dimensional `float32` vector, and each float uses 4 bytes.

$$15{,}000{,}000 \times 8 = 120{,}000{,}000\text{ vectors}$$

$$120{,}000{,}000 \times 1024 \times 4 = 491{,}520{,}000{,}000\text{ bytes}$$

That is **491.52 GB decimal**, or about **457.76 GiB**, for raw vectors alone. It excludes ANN graph/lists, IDs, metadata, replicas, allocator overhead, and temporary index-building memory, so production capacity must be higher. One vector per whole document would be only 61.44 GB, but it would provide poor passage-level retrieval and is not the design used here.

In [ ]:
# Raw vector capacity calculation
documents = 15_000_000
chunks_per_document = 8
dimensions = 1_024
bytes_per_float32 = 4

vectors = documents * chunks_per_document
raw_bytes = vectors * dimensions * bytes_per_float32
decimal_gb = raw_bytes / 1_000_000_000
binary_gib = raw_bytes / 1024**3

print(f"Vectors: {vectors:,}")
print(f"Raw float32 storage: {decimal_gb:,.2f} GB ({binary_gib:,.2f} GiB)")

#### How I would reduce memory

I would measure each option against the golden set rather than applying it blindly:

1. **Matryoshka truncation:** index the first 512 or 256 dimensions of an MRL-trained embedding. At 512 dimensions, raw float32 storage halves to 245.76 GB without re-embedding if full vectors were retained.
2. **Quantization:** float16 halves raw storage; int8 scalar quantization cuts it to roughly one quarter (about 122.88 GB here). Rescore the shortlist with full-precision vectors where needed.
3. **Product quantization/on-disk indexes:** compress or move colder vectors from RAM, accepting a measured recall/latency trade-off.
4. **Better chunking and deduplication:** remove boilerplate and duplicates, and avoid unnecessary overlap. Reducing the average from 8 to 6 chunks saves 25% across vectors, indexing, and embedding work.
5. **Tiering and sharding:** keep hot jurisdictions or recent material in faster memory and cold material on disk/object-backed infrastructure.

Replicas improve availability but multiply capacity; they do not reduce the base requirement.

In [ ]:
# Illustrative raw-storage alternatives; excludes index and metadata overhead
alternatives = [
    ("1024-d float32", 1024, 4),
    ("512-d float32", 512, 4),
    ("1024-d float16", 1024, 2),
    ("1024-d int8", 1024, 1),
    ("256-d int8", 256, 1),
]

for name, dims, bytes_per_value in alternatives:
    size_gb = vectors * dims * bytes_per_value / 1_000_000_000
    print(f"{name:16s}: {size_gb:7.2f} GB")

### Q7 — Why might Arabic retrieval be worse than French?

| Possible cause | How I would distinguish it |
| --- | --- |
| **Weaker Arabic representation in model training**: less or lower-quality Arabic/legal contrastive data, especially for cross-language pairs | Compare clean, human-written Arabic and French test sets matched by topic and difficulty. Break results into Arabic→Arabic and Arabic→English. If Arabic remains worse on clean, correctly chunked text—especially cross-lingually—the embedding/reranker is the likely bottleneck. Compare another multilingual model and a strong Arabic-specific baseline. |
| **Tokenization, normalization, or truncation problems**: Arabic may consume more tokens; diacritics, letter variants, right-to-left marks, and attached clitics can fragment text | Log tokens per character, truncation rate, unknown/rare token patterns, and chunk boundary statistics by language. Run controlled tests before/after Unicode normalization and with shorter Arabic chunks. A quality recovery isolates preprocessing or context-length failure. |
| **Corpus/evaluation quality problems**: weak Arabic OCR, broken reading order, poor translations, fewer relevant judgments, or inconsistent labels | Manually audit a stratified sample with an Arabic-speaking legal reviewer. Compare born-digital text with OCR text and calculate OCR character/word error rates. Re-adjudicate Arabic labels blindly. If clean born-digital documents perform well while OCR documents fail, the model is not the primary cause. |

I would also inspect language-specific sparse analyzers and the cross-encoder. End-to-end failure can come from retrieval, fusion, filtering, or reranking; testing the bi-encoder candidates before reranking identifies which stage introduces the gap.

### Q8 — How many candidates can be reranked?

Using the representative timing assumptions supplied by the lesson:

- End-to-end budget: **2000 ms**
- LLM generation: **1200 ms**
- Query embedding + ANN retrieval: **2 + 15 = 17 ms**
- Cross-encoder cost: **8 ms per candidate**

$$\text{reranking budget} = 2000 - 1200 - 17 = 783\text{ ms}$$

$$k = \left\lfloor\frac{783}{8}\right\rfloor = \mathbf{97\ candidates}$$

Therefore the theoretical answer is **97 candidates** when scores are costed serially using the lesson's timings. In production I would benchmark batch latency on the real GPU and reserve margin for network, fusion, filters, context building, and tail latency; the configured value may therefore be lower (for example 75–90) to protect the p95 two-second SLO.

In [ ]:
# Latency-budget calculation
total_budget_ms = 2_000
llm_generation_ms = 1_200
query_embedding_ms = 2
ann_search_ms = 15
reranker_ms_per_candidate = 8

reranking_budget_ms = (
    total_budget_ms
    - llm_generation_ms
    - query_embedding_ms
    - ann_search_ms
)
candidate_count = reranking_budget_ms // reranker_ms_per_candidate
unused_ms = reranking_budget_ms % reranker_ms_per_candidate

print(f"Reranking budget: {reranking_budget_ms} ms")
print(f"Maximum candidates: {candidate_count}")
print(f"Unused budget under these assumptions: {unused_ms} ms")

## 4. Final recommendation

Start with a benchmark of BGE-M3, multilingual E5, and a Jina multilingual candidate. Deploy the measured winner as the bi-encoder in a hybrid dense+sparse retriever, fuse and ACL-filter results, rerank approximately the top 97 with a multilingual cross-encoder, and pass only the best 8–12 diverse chunks to the LLM.

The most important safeguards are a per-language legal golden set, correct prefixes, token-limit checks, versioned embeddings/indexes, Arabic-specific quality monitoring, strict document permissions, and enough latency headroom to meet the p95 target—not merely the average.